## Chimp → Human Parallel Transport (Kanton et al. 2019)

**Paper:** Kanton et al. 2019, *Organoid single-cell genomic atlas uncovers human-specific features of brain development.* Nature 574, 418–422.

**Question:** Can the chimpanzee organoid trajectory predict human organoid dynamics?

**Strategy:** Use the **chimpanzee time-point sequence** as the control trajectory and **human@earliest bin** as `cf_0`. `reconstruct_cf` propagates that initial human cloud forward using chimp dynamics, producing a predicted human trajectory at later timepoints.

**Prerequisite:** Run `preprocess.ipynb` first to download the data from ArrayExpress (E-MTAB-7552) and save `data/kanton/kanton_preprocessed.h5ad`.

### Timepoint binning

Human and chimp have slightly different day schedules, so cells are binned into:
| Bin | Chimp days | Human days |
|-----|-----------|------------|
| `early` | 31 | 32 |
| `mid` | 61, 64, 69, 71, 74, 80 | 60, 64, 65, 67 |
| `late` | 120 | 120, 128 |

No condition stratification is needed (single organoid culture condition).

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import ot as pot
from src.pt import *
from src.cf_recon import reconstruct_cf

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

out_dir = Path('../../outputs/kanton')
out_dir.mkdir(parents=True, exist_ok=True)

%load_ext autoreload
%autoreload 2

print('Libraries loaded.')

def w2_dist(A, B, max_pts=np.inf, seed=0):
    """W2 distance between two EmpiricalMeasures (or plain numpy arrays).
    Subsamples if clouds exceed max_pts."""
    rng = np.random.default_rng(seed)
    if isinstance(A, EmpiricalMeasure):
        xa, wa = np.asarray(A.locs, float), np.asarray(A.weights, float)
    else:
        xa = np.asarray(A, float); wa = np.ones(len(xa)) / len(xa)
    if isinstance(B, EmpiricalMeasure):
        xb, wb = np.asarray(B.locs, float), np.asarray(B.weights, float)
    else:
        xb = np.asarray(B, float); wb = np.ones(len(xb)) / len(xb)
    wa /= wa.sum(); wb /= wb.sum()
    if len(xa) > max_pts:
        idx = rng.choice(len(xa), int(max_pts), replace=False, p=wa)
        xa, wa = xa[idx], wa[idx]; wa /= wa.sum()
    if len(xb) > max_pts:
        idx = rng.choice(len(xb), int(max_pts), replace=False, p=wb)
        xb, wb = xb[idx], wb[idx]; wb /= wb.sum()
    M = pot.dist(xa, xb, metric='euclidean')
    return np.sqrt(pot.emd2(wa, wb, M, numItermax=1e10))

### Load data

We expect a preprocessed AnnData at `../../data/kanton/kanton_preprocessed.h5ad`.

Required fields:
- `adata.obs['species']`: `'Human'` or `'Chimpanzee'`
- `adata.obs['timepoint_days']`: integer days in culture (e.g. 31, 60, 64, 69, 89, 120)
- `adata.obsm['X_pca']`: PCA coordinates (computed below if missing)

In [ ]:
DATA_DIR  = Path('../../data/kanton')
H5AD_PATH = DATA_DIR / 'kanton_preprocessed.h5ad'

if H5AD_PATH.exists():
    adata_full = sc.read_h5ad(H5AD_PATH)
    print(f'Loaded {adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes')
else:
    raise FileNotFoundError(
        f'{H5AD_PATH} not found.\n'
        'Please run notebooks/kanton/preprocess.ipynb first.'
    )

# Notebook-specific relabel: move chimp day 89 from late -> mid.
mask_day89_chimp = (adata_full.obs['species'] == 'Chimpanzee') & (adata_full.obs['timepoint_days'] == 89)
if mask_day89_chimp.any():
    if hasattr(adata_full.obs['timepoint'], 'cat') and 'mid' not in adata_full.obs['timepoint'].cat.categories:
        adata_full.obs['timepoint'] = adata_full.obs['timepoint'].cat.add_categories(['mid'])
    adata_full.obs.loc[mask_day89_chimp, 'timepoint'] = 'mid'
    print(f"Reassigned {int(mask_day89_chimp.sum())} chimp day-89 cells: late -> mid")
else:
    print('No chimp day-89 cells found to relabel.')

print('\nobs columns:', list(adata_full.obs.columns))
print('obsm keys:  ', list(adata_full.obsm.keys()))
print()
print(adata_full.obs.groupby(['species', 'timepoint'], observed=True).size().to_string())

In [ ]:
# ── Optional fast subsample for quick iteration ───────────────────────────────
SUBSAMPLE_ENABLED = False
SUBSAMPLE_MAX_PER_GROUP = 2000  # max cells per (species, timepoint) bin
SUBSAMPLE_SEED = 42

if SUBSAMPLE_ENABLED:
    rng = np.random.default_rng(SUBSAMPLE_SEED)
    obs = adata_full.obs.reset_index(drop=False)
    sampled_pos = []

    for _, grp in obs.groupby(['species', 'timepoint'], observed=True):
        pos = grp.index.to_numpy()
        if len(pos) > SUBSAMPLE_MAX_PER_GROUP:
            pos = rng.choice(pos, size=SUBSAMPLE_MAX_PER_GROUP, replace=False)
        sampled_pos.append(pos)

    sampled_pos = np.concatenate(sampled_pos)
    sampled_pos = np.sort(sampled_pos)
    adata = adata_full[sampled_pos].copy()

    print(f'Using subsampled adata: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
    print('Counts after subsample:')
    print(adata.obs.groupby(['species', 'timepoint'], observed=True).size().to_string())
else:
    adata = adata_full
    print('Subsampling disabled; using full dataset adata.')
    print(f'Using full adata: {adata.n_obs:,} cells x {adata.n_vars:,} genes')

### PCA setup

Standardize PCA coordinates and identify shared timepoints between human and chimp.

In [ ]:
# ── PCA setup ─────────────────────────────────────────────────────────────────
n_pcs = 15

if 'X_pca' not in adata.obsm:
    print('X_pca not found – running standard scanpy preprocessing ...')
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key='species')
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
    adata.write_h5ad(H5AD_PATH)
    print('Preprocessing done – saved to', H5AD_PATH)

pca_raw = adata.obsm['X_pca'][:, :n_pcs]
pca_std = pca_raw.std(axis=0)
pca     = pca_raw / pca_std

# Use reset index so boolean masks work correctly
_obs = adata.obs.reset_index(drop=True)
_pca = pca

# ── Timepoint ordering ────────────────────────────────────────────────────────
# timepoint column is a string bin: 'early' / 'mid' / 'late'
# (as set by preprocess.ipynb; adjust if your data uses different labels)
TP_ORDER = [tp for tp in ['early', 'mid', 'late']
            if tp in _obs['timepoint'].unique()]
# Representative day numbers for axis labels
TP_DAYS  = {'early': 35, 'mid': 65, 'late': 110}

# Verify both species have cells at each timepoint
chimp_tps = set(_obs.loc[_obs['species'] == 'Chimpanzee', 'timepoint'].unique())
human_tps = set(_obs.loc[_obs['species'] == 'Human',      'timepoint'].unique())
TP_ORDER  = [tp for tp in TP_ORDER if tp in chimp_tps and tp in human_tps]
print(f'Shared timepoint bins: {TP_ORDER}')

# ── Colour scheme ─────────────────────────────────────────────────────────────
COL_CHIMP     = '#D62828'
COL_HUMAN_OBS = '#2E86AB'
COL_PREDICTED = '#2DC653'

CATEGORIES = ['chimp', 'predicted human', 'human']
CAT_COLORS  = {
    'chimp':           COL_CHIMP,
    'predicted human': COL_PREDICTED,
    'human':           COL_HUMAN_OBS,
}

import matplotlib.cm as cm
tp_cmap   = cm.get_cmap('plasma', max(len(TP_ORDER), 2))
TP_COLORS = {tp: tp_cmap(i / max(1, len(TP_ORDER) - 1)) for i, tp in enumerate(TP_ORDER)}
GREY      = '#cccccc'

def get_pca(species, tp):
    mask = (_obs['species'] == species) & (_obs['timepoint'] == tp)
    return _pca[mask.values]

print('\nPoint cloud sizes:')
for sp in ['Chimpanzee', 'Human']:
    for tp in TP_ORDER:
        n = len(get_pca(sp, tp))
        label = f'~day {TP_DAYS.get(tp, tp)}'
        if n > 0:
            print(f'  {sp:12s}  {tp:6s} ({label}): {n}')

### Run WPT

1. `control = [chimp@tp0, chimp@tp1, ...]` — chimp trajectory drives the transport
2. `cf_0    = human@tp0` — initial human distribution (earliest shared timepoint)
3. `reconstruct_cf(control, cf_0)` → predicted human curve at subsequent timepoints

`pred_curve[0]` == `cf_0` (trivially exact); genuine predictions are at later timepoints.

In [ ]:
chimp_traj = [get_pca('Chimpanzee', tp) for tp in TP_ORDER]
human_obs  = [get_pca('Human',       tp) for tp in TP_ORDER]

# Keep only bins where both species have cells
valid = [(tp, c, h) for tp, c, h in zip(TP_ORDER, chimp_traj, human_obs)
         if len(c) > 0 and len(h) > 0]
TP_ORDER   = [v[0] for v in valid]
chimp_traj = [v[1] for v in valid]
human_obs  = [v[2] for v in valid]

print(f'{len(TP_ORDER)} shared bins with cells in both species: {TP_ORDER}')
for tp, c, h in zip(TP_ORDER, chimp_traj, human_obs):
    print(f'  {tp:6s} (~day {TP_DAYS.get(tp,"?"):3}): Chimp={len(c)}, Human={len(h)}')

cf0     = human_obs[0]
n_steps = len(TP_ORDER) - 1

print(f'\ncf_0 = Human @ {TP_ORDER[0]}: {len(cf0)} cells')
print(f'Running reconstruct_cf ({n_steps} steps, {n_pcs} PCs) ...')
pred_curve = reconstruct_cf(chimp_traj, cf0, n=n_steps, project=False, tol=1e-6)
print('Done.')

human_obs_em = [EmpiricalMeasure(h, np.ones(len(h)) / len(h)) for h in human_obs]
chimp_em     = [EmpiricalMeasure(c, np.ones(len(c)) / len(c)) for c in chimp_traj]

# Pin pred_curve[0] to exact human@tp0 (avoids np.unique reordering artefact)
pred_curve[0] = human_obs_em[0]

### Trajectory visualisation (PC1–PC2)

In [ ]:
def wmean(em):
    return (em.weights[:, None] * em.locs).sum(0)

c_means = np.array([wmean(c) for c in chimp_em])
h_means = np.array([wmean(h) for h in human_obs_em])
p_means = np.array([wmean(p) for p in pred_curve])

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(c_means[:, 0], c_means[:, 1], 'o-',
        color=COL_CHIMP,     lw=2.5, ms=8, label='Chimpanzee (observed)')
ax.plot(h_means[:, 0], h_means[:, 1], 's-',
        color=COL_HUMAN_OBS, lw=2.5, ms=8, label='Human (observed)')
ax.plot(p_means[:, 0], p_means[:, 1], '^--',
        color=COL_PREDICTED,  lw=2.5, ms=8, label='Human (predicted via WPT)')

for i, tp in enumerate(TP_ORDER):
    lbl = f'{tp}\n(~day {TP_DAYS.get(tp, tp)})'
    ax.annotate(lbl, c_means[i, :2], textcoords='offset points',
                xytext=(5, 5), fontsize=8, color='#555555')

ax.set_xlabel('PC 1 (standardized)', fontsize=12)
ax.set_ylabel('PC 2 (standardized)', fontsize=12)
ax.set_title('Chimp-driven WPT vs observed Human (Kanton 2019)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'kanton_wpt_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

### Point-cloud scatter at each time point

In [ ]:
fig, axes = plt.subplots(1, len(TP_ORDER),
                          figsize=(7 * len(TP_ORDER), 6),
                          sharex=False, sharey=False)
if len(TP_ORDER) == 1: axes = [axes]

for ax, tp, c, h, p in zip(axes, TP_ORDER, chimp_em, human_obs_em, pred_curve):
    ax.scatter(c.locs[:, 0], c.locs[:, 1], c=COL_CHIMP,     s=4, alpha=0.4,
               label=f'Chimp obs  (n={len(c.locs)})')
    ax.scatter(h.locs[:, 0], h.locs[:, 1], c=COL_HUMAN_OBS, s=4, alpha=0.4,
               label=f'Human obs  (n={len(h.locs)})')
    ax.scatter(p.locs[:, 0], p.locs[:, 1], c=COL_PREDICTED, s=4, alpha=0.4,
               label=f'Predicted  (n={len(p.locs)})')
    for em, col in [(c, COL_CHIMP), (h, COL_HUMAN_OBS), (p, COL_PREDICTED)]:
        cx = (em.weights[:, None] * em.locs).sum(0)
        ax.scatter(cx[0], cx[1], c=col, s=200, marker='*',
                   edgecolors='k', linewidths=0.8, zorder=5)
    ax.set_xlabel('PC 1', fontsize=11); ax.set_ylabel('PC 2', fontsize=11)
    ax.set_title(f'{tp}  (~day {TP_DAYS.get(tp, tp)})', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, markerscale=3); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'kanton_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### W2 table across PCA dimensions (`n_pcs`)

This block evaluates W2 to observed human at each timepoint for:
- WPT prediction
- Mean-shift baseline

for `n_pcs in [5, 10, 15, 20]`, and saves a LaTeX table.

In [ ]:
n_pcs_values = [2, 3, 5, 7, 10, 15, 20]
tp_bins = ['early', 'mid', 'late']
TP_DAYS = {'early': 35, 'mid': 65, 'late': 110}

if 'X_pca' not in adata.obsm:
    raise ValueError("adata.obsm['X_pca'] is required for this sweep.")

obs_df = adata.obs.reset_index(drop=True)
results = {}

def weighted_mean(em):
    return (em.weights[:, None] * em.locs).sum(0)

for n_pcs_eval in n_pcs_values:
    print(f"Evaluating n_pcs={n_pcs_eval} ...")
    pca_raw_eval = adata.obsm['X_pca'][:, :n_pcs_eval]
    pca_std_eval = pca_raw_eval.std(axis=0)
    pca_std_eval[pca_std_eval == 0] = 1.0
    pca_eval = pca_raw_eval / pca_std_eval

    def get_pca_eval(species, tp):
        mask = (obs_df['species'] == species) & (obs_df['timepoint'] == tp)
        return pca_eval[mask.values]

    tp_order_eval = [tp for tp in tp_bins if tp in obs_df['timepoint'].unique()]
    chimp_tps = set(obs_df.loc[obs_df['species'] == 'Chimpanzee', 'timepoint'].unique())
    human_tps = set(obs_df.loc[obs_df['species'] == 'Human', 'timepoint'].unique())
    tp_order_eval = [tp for tp in tp_order_eval if tp in chimp_tps and tp in human_tps]

    chimp_traj_eval = [get_pca_eval('Chimpanzee', tp) for tp in tp_order_eval]
    human_obs_eval = [get_pca_eval('Human', tp) for tp in tp_order_eval]

    valid = [(tp, c, h) for tp, c, h in zip(tp_order_eval, chimp_traj_eval, human_obs_eval)
             if len(c) > 0 and len(h) > 0]
    tp_order_eval = [v[0] for v in valid]
    chimp_traj_eval = [v[1] for v in valid]
    human_obs_eval = [v[2] for v in valid]

    if len(tp_order_eval) < 2:
        raise ValueError(f"Need at least 2 shared bins for n_pcs={n_pcs_eval}, got {tp_order_eval}")

    cf0_eval = human_obs_eval[0]
    n_steps_eval = len(tp_order_eval) - 1
    pred_curve_eval = reconstruct_cf(chimp_traj_eval, cf0_eval, n=n_steps_eval, project=False, tol=1e-6)

    human_obs_em_eval = [EmpiricalMeasure(h, np.ones(len(h)) / len(h)) for h in human_obs_eval]
    chimp_em_eval = [EmpiricalMeasure(c, np.ones(len(c)) / len(c)) for c in chimp_traj_eval]
    pred_curve_eval[0] = human_obs_em_eval[0]

    chimp_means_eval = np.array([weighted_mean(c) for c in chimp_em_eval])
    mean_shifts_eval = [chimp_means_eval[i + 1] - chimp_means_eval[i] for i in range(len(tp_order_eval) - 1)]
    shift_samples_eval = [human_obs_eval[0].copy()]
    for i in range(len(tp_order_eval) - 1):
        shift_samples_eval.append(shift_samples_eval[-1] + mean_shifts_eval[i])
    shift_em_eval = [EmpiricalMeasure(x, np.ones(len(x)) / len(x)) for x in shift_samples_eval]

    d_wpt, d_shift = [], []
    for i, tp in enumerate(tp_order_eval):
        h = human_obs_em_eval[i]
        p_wpt = pred_curve_eval[i]
        p_shift = shift_em_eval[i]
        d_wpt.append(w2_dist(p_wpt, h))
        d_shift.append(w2_dist(p_shift, h))

    results[n_pcs_eval] = dict(tp_order=tp_order_eval, wpt=d_wpt, shift=d_shift)
    print(f"  done: bins={tp_order_eval}")

# Build LaTeX table (exclude IC bin 'early' / day 35).
all_tps = [
    tp for tp in tp_bins
    if tp != 'early' and all(tp in results[n]['tp_order'] for n in n_pcs_values)
]
if not all_tps:
    raise ValueError("No non-initial common timepoint bins found across all n_pcs runs.")

lines = []
col_spec = 'c' + 'cc' * len(all_tps)
lines.append(r'\begin{table}[h]')
lines.append(r'\centering')
lines.append(r'\small')
lines.append(r'\begin{tabular}{' + col_spec + '}')
lines.append(r'\toprule')

hdr1 = [rf'\multicolumn{{2}}{{c}}{{{tp} (~day {TP_DAYS.get(tp,tp)})}}' for tp in all_tps]
lines.append(r'$n_{\mathrm{pcs}}$ & ' + ' & '.join(hdr1) + r' \\')

cmr = ''.join([rf'\cmidrule(lr){{{2+2*i}-{3+2*i}}}' for i in range(len(all_tps))])
lines.append(cmr)

hdr2 = []
for _ in all_tps:
    hdr2.extend([r'WPT', r'Mean-shift'])
lines.append(' & ' + ' & '.join(hdr2) + r' \\')
lines.append(r'\midrule')

for n in n_pcs_values:
    row = [str(n)]
    tp_to_idx = {tp: i for i, tp in enumerate(results[n]['tp_order'])}
    for tp in all_tps:
        j = tp_to_idx[tp]
        row.append(f"{results[n]['wpt'][j]:.3f}")
        row.append(f"{results[n]['shift'][j]:.3f}")
    lines.append(' & '.join(row) + r' \\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(
    r'\caption{$W_2$ to observed human per non-initial timepoint for WPT and mean-shift baseline across PCA dimensions.}'
)
lines.append(r'\label{tab:kanton_w2_npcs}')
lines.append(r'\end{table}')

latex_table = '\n'.join(lines)
out_path = out_dir / 'kanton_w2_npcs_table.tex'
out_path.write_text(latex_table)

print('\n' + '=' * 60)
print(latex_table)
print(f'\nSaved -> {out_path}')

### UMAP: group-coloured 3-pane plot

Joint UMAP embedding on all three groups × all timepoints.  
One pane per group (chimp / predicted human / human); the other two are shown in grey.

In [ ]:
import umap as umap_lib

all_locs, all_labels, all_tps = [], [], []

for i, tp in enumerate(TP_ORDER):
    c = chimp_em[i]; h = human_obs_em[i]; p = pred_curve[i]
    all_locs.append(c.locs); all_labels.extend(['chimp']           * len(c.locs))
    # if we are at first timepoint, skip human_obs to avoid duplicate points (pred_curve[0] == human_obs_em[0])
    if tp == "early":
        all_locs.append(h.locs); all_labels.extend(['human']           * len(h.locs))
        all_tps.extend([tp] * (len(c.locs) + len(h.locs)))
    else:
        all_locs.append(h.locs); all_labels.extend(['human']           * len(h.locs))
        all_locs.append(p.locs); all_labels.extend(['predicted human'] * len(p.locs))
        all_tps.extend([tp] * (len(c.locs) + len(h.locs) + len(p.locs)))

X_all   = np.vstack(all_locs)
labels  = np.array(all_labels)
tps_arr = np.array(all_tps)

print(f'Fitting UMAP on {len(X_all):,} points x {X_all.shape[1]} dims ...')
reducer   = umap_lib.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
embedding = reducer.fit_transform(X_all)
print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, cat in zip(axes, CATEGORIES):
    mask_hi = labels == cat
    ax.scatter(embedding[~mask_hi, 0], embedding[~mask_hi, 1],
               c=GREY, s=2, alpha=0.3, linewidths=0, rasterized=True)
    ax.scatter(embedding[mask_hi, 0],  embedding[mask_hi, 1],
               c=CAT_COLORS[cat], s=4, alpha=0.6, linewidths=0, rasterized=True)
    ax.set_title(cat.capitalize(), fontsize=13, fontweight='bold',
                 color=CAT_COLORS[cat])
    ax.set_xlabel('UMAP 1', fontsize=11); ax.set_ylabel('UMAP 2', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

plt.suptitle('Kanton 2019 — UMAP by group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'kanton_umap_groups.png', dpi=150, bbox_inches='tight')
plt.show()

### UMAP: 3 × N grid (rows = group, cols = timepoint)

Each pane shows only the group × timepoint intersection in colour; everything else is grey.

In [ ]:
n_tp   = len(TP_ORDER)
n_cats = len(CATEGORIES)
fig, axes = plt.subplots(n_cats, n_tp, figsize=(6 * n_tp, 6 * n_cats))

if n_cats == 1: axes = axes[np.newaxis, :]
if n_tp   == 1: axes = axes[:, np.newaxis]

# Override final timepoint color to avoid yellow from the default colormap.
if TP_ORDER:
    TP_COLORS[TP_ORDER[-1]] = '#2A9D8F'  # teal

for row, cat in enumerate(CATEGORIES):
    for col, tp in enumerate(TP_ORDER):
        ax   = axes[row, col]
        mask = (labels == cat) & (tps_arr == tp)
        # augment predicted human early timepoint because it overlaps exactly with human obs early (same points, same labels)
        if cat == 'predicted human' and tp == 'early':
            mask |= (labels == 'human') & (tps_arr == 'early')
        ax.scatter(embedding[~mask, 0], embedding[~mask, 1],
                   c=GREY, s=2, alpha=0.3, linewidths=0, rasterized=True)
        if mask.sum() > 0:
            tp_col = TP_COLORS[tp]
            ax.scatter(embedding[mask, 0], embedding[mask, 1],
                       c=[tp_col], s=4, alpha=0.7, linewidths=0, rasterized=True)

        if row == 0:
            day_lbl = f'{tp}\n(~day {TP_DAYS.get(tp,tp)})'
            ax.set_title(day_lbl, fontsize=12, fontweight='bold',
                         color=TP_COLORS[tp])
        if col == 0:
            ax.set_ylabel(cat.capitalize(), fontsize=12, fontweight='bold',
                          color=CAT_COLORS[cat])
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

plt.tight_layout()
plt.savefig(out_dir / 'kanton_umap_grid.png', dpi=150, bbox_inches='tight')
plt.show()